<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7-white-bg.png">

# Lab: Train and Evaluate the Conversational Model

In this second lab, we will train and evaluate our conversational model.


## Overview

In this lab, you will:

* Load the preprocessed dataset prepared in the first lab.

* Configure the Gemma model and LoRA hyperparameters.

* Execute the fine-tuning process.

* Test the fine-tuned model and run manual spot checks on unseen data.

* Iterate on the hyperparameter settings and fine-tuning process until we are satisfied with the model's behavior.

The end result of this lab will be a conversational model that can respond to user queries as specified in the task description.

## Imports

Import a range of libraries that we may use for fine-tuning and evaluating the model.

In [ ]:
# Standard library imports.
import io # Input/output operations.
import json # JSON data handling.
import os # Operating system interface.
import random # For random number generation and shuffling of data.
import re # Regular expressions for text processing.
import unicodedata # Unicode character database.
from textwrap import fill  # Text wrapping utilities.
from typing import Any, Dict, Tuple, List

# Third-party data and utility libraries.
import numpy as np # Numerical computing.
import pandas as pd # Data manipulation and analysis.
from tqdm import tqdm # Progress bars.

import matplotlib.pyplot as plt # Plotting and visualisation.

# Set Keras and JAX settings.
os.environ["KERAS_BACKEND"] = "jax"

# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Deep learning and AI libraries.
import jax.numpy as jnp # JAX numerical computing.
import keras # High-level neural networks API.
import keras_nlp # Keras NLP extensions.

# Google Colab specific.
from google.colab import drive # Access to Google Drive.
from google.colab import userdata # Access to Colab secrets.

##  1: Load the data

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive/` to the beginning of your file path.

In [ ]:
# Connect this notebook to Google Drive.
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load files from Google Drive into dataframes
train_df = pd.read_json("/content/drive/MyDrive/Colab Notebooks/conversational_training_data.jsonl", lines=True)
test_df = pd.read_json("/content/drive/MyDrive/Colab Notebooks/conversational_test_data.jsonl", lines=True)

# Transform dataframes into lists of dictionaries
train_data = train_df.to_dict("records")
print(train_data[0])

test_data = test_df.to_dict("records")
print(test_data[0])


{'history': [], 'input': 'Is [-43,29] right for [[3,-4],[-3,2]] times [-5,7]?', 'output': 'Right — [-43,29] is correct. Nice — can you say in one line why row one: 3·-5 + (-4)·7; row two: (-3)·-5 + 2·7 was the right move here?'}
{'history': [], 'input': "I'm exhausted — please just give me the answer for [7,7] and [-1,-6].", 'output': "It's [6,1]: 7+(-1) and 7+(-6). Think of it as a step-by-step move on a game board, not a fixed square."}


### Configure Kaggle API key for Gemma

To access Gemma models, we need to provide our Kaggle credentials.

#### Load the Kaggle username and key

In [ ]:
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## 2: Preparation for fine-tuning


### Prepare data for Gemma3 LoRA fine-tuning

### Format conversations with special tokens

Now that the data is in plain `history`/`input`/`output` form, this cell
adds the special tokens the model needs to recognize turn boundaries.

**What this cell does:**
* Defines `format_io_as_turns()`, which wraps a single question/answer
  pair with `<start_of_turn>`/`<end_of_turn>` tokens — the same markers
  Gemma was originally trained to recognize as conversation turns.
* Defines `format_history()`, which applies those same tokens to every
  prior turn in a conversation, so multi-turn context is represented
  with real turn boundaries rather than plain text.
* For each training example, formats the current question/answer pair,
  then **prepends the formatted history in front of it** so multi-turn
  examples carry their full context, with every turn (past and current)
  correctly tagged and in the right order.
* Collects the results into two parallel lists, `inputs` (prompts) and
  `targets` (responses), and prints the first example to confirm the
  formatting looks correct before training.
* Packages both lists into a `data` dictionary with `"prompts"` and
  `"responses"` keys — the format the Gemma3 Keras model expects.

In [ ]:
SOT = "<start_of_turn>"
EOT = "<end_of_turn>"

# Format inputs and outputs
def format_io_as_turns(
    input_text: str, output_text: str, sot: str = SOT, eot: str = EOT
) -> tuple[str, str]:
    formatted_q = f"{sot}user\n{input_text}{eot}"
    formatted_a = f"{sot}model\n{output_text}{eot}"
    return formatted_q, formatted_a

# Format history
def format_history(history: list, sot: str = SOT, eot: str = EOT) -> str:
    text = ""
    for turn in history:
        role = "user" if turn["role"] == "student" else "model"
        text += f"{sot}{role}\n{turn['content']}{eot}\n"
    return text

inputs  = []
targets = []

for example in train_data:
  q, a =  format_io_as_turns(example["input"], example["output"])
  q = format_history(example["history"]) + q
  inputs.append(q)
  targets.append(a)

print(inputs[0])
print(targets[0])

# Create "data" dictionary with "prompts" and "responses" data
data = {"prompts": inputs, "responses": targets}

<start_of_turn>user
Is [-43,29] right for [[3,-4],[-3,2]] times [-5,7]?<end_of_turn>
<start_of_turn>model
Right — [-43,29] is correct. Nice — can you say in one line why row one: 3·-5 + (-4)·7; row two: (-3)·-5 + 2·7 was the right move here?<end_of_turn>


### Load Gemma using Keras
Once the data is ready, we now need to fine-tune your Gemma model using LoRA. We will train only a small number of additional parameters whilst keeping the base model frozen.
You may find additional information in [Google's *Fine-tune Gemma in Keras using LoRA* documentation](https://ai.google.dev/gemma/docs/core/lora_tuning).

In [ ]:
keras.utils.set_random_seed(812)  # Replicate randomness in Keras.

# Load Gemma3-1B model paramaters
model = keras_nlp.models.Gemma3CausalLM.from_preset(preset="gemma3_1b")
model.summary()

100%|██████████| 966/966 [00:00<00:00, 1.95MB/s]


100%|██████████| 3.23k/3.23k [00:00<00:00, 4.91MB/s]


100%|██████████| 4.47M/4.47M [00:00<00:00, 8.59MB/s]


100%|██████████| 1.86G/1.86G [00:36<00:00, 55.0MB/s]


Preprocessor: "gemma3_causal_lm_preprocessor"

...

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 1152)        │     999,885,952 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     301,989,888 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 999,885,952 (3.72 GB)

 Trainable params: 999,885,952 (3.72 GB)

 Non-trainable params: 0 (0.00 B)

### Evaluate the base model's performance

Before fine-tuning, this is a good moment to **test the base (foundation) Gemma model** with a small set of example prompts from the training data.
We can **reuse these same prompts after fine-tuning** to see how LoRA adaptation changes the model's conversational responses, giving us a direct, side-by-side comparison of performance.

The code below shuffles the test data (some of the prompt-output pairs that you constructed in the first lab) and generates a response for the first five examples in the test set.

In [ ]:
# Shuffle the data.
random.shuffle(test_data)

print("Testing baseline prompts on the base model:\n")

# Test the base model on a shuffled sample.
for entry in test_data[:5]:
    prompt = entry["input"]
    expected_response = entry["output"]
    print(f"Prompt: {prompt}")
    print(f"Sample of an expected response: {expected_response[:150]}...")
    print(f"Base model response: {model.generate(prompt, max_length=500)}")
    print("-" * 50)

Testing baseline prompts on the base model:

Prompt: So for [-3,-3] onto [0,1], keep only the part of the vector lying along [0,1] gives [0,-3]?
Sample of an expected response: Right — [0,-3] is correct. Confirmed. What made keep only the part of the vector lying along [0,1] the correct operation instead of something else?...
Base model response: So for [-3,-3] onto [0,1], keep only the part of the vector lying along [0,1] gives [0,-3]?

I'm not sure if I'm understanding the question correctly.

I think you're looking for the vector that is perpendicular to the line.

The vector that is perpendicular to the line is the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that is perpendicular to the vector that i

> **Note: Understanding base model limitations**
>
> The base Gemma3-1B model produces inconsistent or off-topic responses when asked to answer conversational prompts.
>
> **Base models vs instruction-tuned models**:
>
> The Gemma3-1B model used here is a **base (pretrained) model**, not an **instruction-tuned** variant. Base models are trained solely on **next-token prediction**, learning to predict the most likely next word given the preceding context. They have not been further trained to interpret and follow human instructions.
>
>
> **Why small models struggle more**:
>
> With only 1 billion parameters, the Gemma3-1B model has limited capacity to capture complex patterns and implicit task instructions from context alone. Larger models can sometimes infer task requirements from prompt phrasing, but smaller base models are more prone to producing off-topic or hallucinated responses, generating text that sounds plausible but is factually incorrect or irrelevant to the question asked.
>


## 3: Activate and run LoRA


### Set hyperparameters and enable LoRA

Before training starts, this cell fixes the key hyperparameters and turns
on LoRA so only a small set of adapter weights gets trained.

**What this cell does, and why these values:**
* `lora_rank = 8`: kept low since this fine-tune is teaching a
  behavior/personality on top of math the base model already knows,
  not new knowledge. A small rank is enough capacity for that, and
  lowers the risk of overfitting on a relatively small (~1,000 example)
  dataset.
* `learning_rate = 5e-5`: a conservative starting point appropriate for
  LoRA on a dataset this size; high enough to learn the target behavior,
  low enough to avoid overwriting it too aggressively in a few epochs.
* `num_epochs = 5`: enough passes to let the behavior settle in without
  assuming more training is automatically better; intended to be
  monitored (via generations/loss) rather than treated as fixed.
* `batch_size = 1`: set by hardware memory limits; a small batch keeps
  memory usage low and stable during training.
* `sequence_length = 450`: matches the length used to build the
  training data, so no conversation gets truncated during fine-tuning.
* Calls `enable_lora(rank=lora_rank)` to actually attach the LoRA
  adapters to the model, then prints the optimizer's learning rate as a
  quick sanity check that everything is wired up correctly before
  training starts.

In [ ]:
# Set the hyperparameters and enable LoRA training
lora_rank = 8
learning_rate = 5e-5
num_epochs = 5
batch_size = 1
sequence_length = 450

# Activate LoRA
model.backbone.enable_lora(rank=lora_rank)

# Check learning rate
print(model.optimizer.learning_rate)


<Variable path=adam/learning_rate, shape=(), dtype=float32, value=1.9999999494757503e-05>


### Fine-tuning

The training loop applies the hyperparameters set
above, then trains one epoch at a time so we can watch quality evolve
rather than only seeing a single result at the end.

After each epoch, it generates on the fixed sample prompts and prints the
prompt, the expected (training) response, what the model actually
generated, and whether it starts degrading in later epochs (an early sign of overfitting).

In [ ]:
model.optimizer.learning_rate = learning_rate
model.preprocessor.sequence_length = sequence_length

sample_records = [test_data[0]]
multi_turn_sample = next((r for r in test_data if r["history"]), None)
if multi_turn_sample:
    sample_records.append(multi_turn_sample)

def build_generation_prompt(record, sot=SOT, eot=EOT):
    history_text = format_history(record["history"])
    q, _ = format_io_as_turns(record["input"], "")
    return history_text + q + f"\n{sot}model\n"

for i in jnp.arange(num_epochs):
    print("\n\nEpoch:" + str(i + 1) + "\n")
    model.fit(data, epochs=1, batch_size=batch_size, verbose=1)

    print(f"\nSample generations after epoch {i + 1}")
    for rec in sample_records:
        prompt = build_generation_prompt(rec)
        generated = model.generate(prompt, max_length=sequence_length)

        new_text = generated[len(prompt):] if generated.startswith(prompt) else generated

        print("\nPROMPT   :", rec["input"])
        print("EXPECTED :", rec["output"][:150])
        print("GENERATED:", new_text[:150])
    print("-" * 50)



Epoch:1

958/958 ━━━━━━━━━━━━━━━━━━━━ 581s 579ms/step - loss: 0.2749 - sparse_categorical_accuracy: 0.5122

Sample generations after epoch 1

PROMPT   : So for [-3,-3] onto [0,1], keep only the part of the vector lying along [0,1] gives [0,-3]?
EXPECTED : Right — [0,-3] is correct. Confirmed. What made keep only the part of the vector lying along [0,1] the correct operation instead of something else?
GENERATED: Right — [-3,-3] onto [0,1] gives [0,-3] — [0,1] is the right answer. What's the next step?<end_of_turn>

PROMPT   : I think it's [13,-15].
EXPECTED : Exactly right, [13,-15]. You've got it — that's the confirmed answer.
GENERATED: Exactly right, [13,-15]. Now — what do you think happens if one of those numbers flips sign?<end_of_turn>
--------------------------------------------------


Epoch:2

958/958 ━━━━━━━━━━━━━━━━━━━━ 566s 591ms/step - loss: 0.1637 - sparse_categorical_accuracy: 0.6565

Sample generations after epoch 2

PROMPT   : So for [-3,-3] onto [0,1], keep only the

### Save Model Weights

The following cell persist the trained LoRA weights to Google Drive so they
survive beyond this runtime session, and verify the save by checking the file exists and reporting its size because a save that silently fails or is interrupted partway (e.g. due to a storage quota limit) can otherwise go unnoticed until it's too late to retrain.

In [ ]:
# Save model weights
model.save_weights("/content/drive/MyDrive/Colab Notebooks/gemma_tutor_epoch5.weights.h5")

In [ ]:
# check that the file was saved successfully
path = "/content/drive/MyDrive/Colab Notebooks/gemma_tutor_epoch5.weights.h5"
print(os.path.exists(path), os.path.getsize(path) / 1e9, "GB")

True 4.011184408 GB


### Reload the fine-tuned model from saved weights

This cell recreates the model architecture from scratch and loads the
trained weights back into it.

In [ ]:
# Reload model weights
model = keras_nlp.models.Gemma3CausalLM.from_preset(preset="gemma3_1b", dtype="bfloat16")
model.backbone.enable_lora(rank=lora_rank)
model.load_weights("/content/drive/MyDrive/Colab Notebooks/gemma_tutor_epoch5.weights.h5")

/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 210 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### Verify the reloaded model actually works

After loading the saved weights into a fresh model, this cell confirms
the reload genuinely worked because a `load_weights()` call can complete
without error while still being subtly wrong, so the only real proof is
generating something and reading it.

In [ ]:
sample_records = [test_data[0]]
multi_turn_sample = next((r for r in test_data if r["history"]), None)
if multi_turn_sample:
    sample_records.append(multi_turn_sample)

def build_generation_prompt(record, sot=SOT, eot=EOT):
    history_text = format_history(record["history"])
    q, _ = format_io_as_turns(record["input"], "")
    return history_text + q + f"\n{sot}model\n"

In [ ]:
prompt = build_generation_prompt(test_data[0])
generated = model.generate(prompt, max_length=sequence_length)
print(generated)

<start_of_turn>user
So for [-3,-3] onto [0,1], keep only the part of the vector lying along [0,1] gives [0,-3]?<end_of_turn>
<start_of_turn>model
Right — [0,-3] is correct. Right instinct — for [-3,-3] onto [0,1], keep only the part of the vector lying along [0,1] gives [0,0]?<end_of_turn>


## 4: Evaluation


This section assesses the model through human review: comparing it
against the base model, and testing how well it generalizes to prompts
it never saw during training.

**What this evaluation covers:**
* **Generation config**: `max_length=600`, higher than the training
  `sequence_length=450`, so responses aren't cut off mid-answer.
* **Evaluation prompts**: a mix of held-out test-set examples (Option B:
  genuinely unseen, since the split happened before any pairs were
  built) plus hand-crafted prompts that deliberately deviate further:
  slight rephrasings, cross-concept questions, format/tone changes, and
  topics outside the 11 trained concepts entirely.
* **Base vs. fine-tuned comparison**: the same prompts run through both
  models, to see what fine-tuning actually changed in tone, structure,
  and domain relevance.
* **Generalization check**: specifically watching whether the model
  applies its trained scaffolding/correction behavior sensibly to novel
  phrasings, or whether it breaks down (loses topic, misses a wrong
  answer, drifts to unrelated content).
* **Manual review**: every response is read and judged qualitatively,
  noting where behavior held up vs. where it broke, rather than relying
  on a single automated score.

In [ ]:
def generate_answer(question: str, history: list | None = None,
                    sot: str = "<start_of_turn>", eot: str = "<end_of_turn>",) -> str:
    history_text = format_history(history) if history else ""
    prompt = (
        f"{sot}user\n"
        f"{question}{eot}\n"
        f"{sot}model\n"
    )

    # With the sequence_length=450, we set max_length to 600 for the model
    # to generate far past what it saw during training
    raw_output = model.generate(prompt, max_length=600)

    # Extract only the newly generated content
    completion = raw_output.replace(prompt, "").strip()

    # Clean up trailing end token
    if completion.endswith(eot):
        completion = completion[:-len(eot)].strip()

    # Return fromatted model response
    return f"{sot}model\n{completion}{eot}"


def ask(question: str) -> str:
    # Generate and return a formatted model response
    formatted = question.strip()
    return generate_answer(formatted)


# Manual spot checks - linear algebra tutor

print(ask("Can you help me understand what an eigenvector actually is?"))
print(ask("I'm stuck on why the determinant of [[2,4],[1,2]] comes out to zero."))
print(ask("What does it mean for two vectors to be orthogonal?"))
print(ask("Is [4,8] just a scaled version of [1,2]?"))
print(ask("Explain the idea behind projecting one vector onto another."))
print(ask("Give me an intuitive way to think about matrix rank."))
print(
    ask(
        "I need to explain span to a classmate who is completely lost. "
        "Can you give me a simple way to describe it?"
    )
)


<start_of_turn>model
Picture a rope stretched between two mountains. The mountain that the rope is closest to is its closest point, and the point that the rope is farthest from is its farthest point. The eigenvector is the closest point, and the eigenvector is the farthest point. What's the eigenvector for the matrix A that scales the vector x by 2?<end_of_turn>
<start_of_turn>model
Think of it as a scorecard for how well two teams' directions matched. Try it: what's 2·1 − 4·2?<end_of_turn>
<start_of_turn>model
Imagine a rope stretched tautly over a pole, and ask: how much of a tug is needed to move the pole? What does that tell you about the two directions?<end_of_turn>
<start_of_turn>model
It's not. It's a scaled copy of the vector [1,2] itself. What makes [4,8] a scaled copy of [1,2]?<end_of_turn>
<start_of_turn>model
Let's picture a projector that casts only on the wall in front of it, not the wall itself. Try [1,2] onto [1,2] — 1·[1,2] + 2·[1,2]. What do you get?<end_of_turn>
<sta

### Comparing base model vs fine-tuned model outputs

Now that the model has been fine-tuned with LoRA, we can test it using the **same prompts** we used earlier to evaluate the base model. This allows for a direct, side-by-side comparison of how fine-tuning has affected the model's conversational behavior.

- **Base model (before fine-tuning)**: produced inconsistent, off-topic, or hallucinated responses because it was trained only on next-token prediction.

- **Fine-tuned model (after LoRA)**: Ideally this will now produce more coherent, contextually relevant responses that match the expected conversational style and domain knowledge, demonstrating how LoRA adaptation has taught the model to generate answers that are closer to the ones you included in the training data.


In [ ]:
# Test the fine-tuned model on seen training examples.
# Deliberately sample both single-turn AND multi-turn entries
random.seed(0)

single_turn_pool = [r for r in train_data if not r["history"]]
multi_turn_pool = [r for r in train_data if r["history"]]

sample = random.sample(single_turn_pool, 3) + random.sample(multi_turn_pool, 2)
random.shuffle(sample)  # mix the order so single/multi aren't grouped

print("Testing model on seen prompts:\n")

for entry in sample:
    prompt_text = entry["input"]
    expected_response = entry["output"]
    is_multi_turn = bool(entry["history"])

    print(f"Prompt: {prompt_text}")
    if is_multi_turn:
        print(f"  (multi-turn: {len(entry['history']) // 2} prior exchange(s) included as context)")
    print(f"Sample of an expected response: {expected_response[:100]}...")
    print(f"Fine-tuned model response: {generate_answer(prompt_text, history=entry['history'])}")
    print("-" * 50)

Testing model on seen prompts:

Prompt: Confirming: [-4,7] and [-9,-4] should give [-13,3]?
Sample of an expected response: Right — [-13,3] is correct. Nice — can you say in one line why -4+(-9) and 7+(-4) was the right move...
Fine-tuned model response: <start_of_turn>model
Right — [-13,3] is correct. Right instinct — for [-4,7] and [-9,-4], what would you expect if the two vectors were swapped?<end_of_turn>
--------------------------------------------------
Prompt: So for [[8,7],[8,3]], 8·3 − 7·8 gives -32?
Sample of an expected response: Right — -32 is correct. Nice — can you say in one line why 8·3 − 7·8 was the right move here?...
Fine-tuned model response: <start_of_turn>model
Right — -32 is correct. Confirmed. What made 8·3 − 7·8 the correct operation instead of something else?<end_of_turn>
--------------------------------------------------
Prompt: For columns [5,-1] and [-6,1], vector [3,4], I think the answer is [15,4].
Sample of an expected response: Try a warm-up: track a si

### Testing generalization with unseen prompts

To better assess whether the fine-tuned model has truly learned the conversational task, we can test it with **new prompts** that were not part of the training data.



In [ ]:
# Test the fine-tuned model on unseen test-set examples
import random
random.seed(1)  # different seed from the seen-prompts test, for a fresh sample

single_turn_test_pool = [r for r in test_data if not r["history"]]
multi_turn_test_pool = [r for r in test_data if r["history"]]

test_sample = random.sample(single_turn_test_pool, 3) + random.sample(multi_turn_test_pool, 2)
random.shuffle(test_sample)

print("Testing model on unseen prompts:\n")

for entry in test_sample:
    prompt_text = entry["input"]
    expected_response = entry["output"]
    is_multi_turn = bool(entry["history"])

    print(f"Prompt: {prompt_text}")
    if is_multi_turn:
        print(f"  (multi-turn: {len(entry['history']) // 2} prior exchange(s) included as context)")
    print(f"Sample of an expected response: {expected_response[:100]}...")
    print(f"Fine-tuned model response: {generate_answer(prompt_text, history=entry['history'])}")
    print("-" * 50)

Testing model on unseen prompts:

Prompt: [3,2].
  (multi-turn: 1 prior exchange(s) included as context)
Sample of an expected response: Exactly. So subtraction is just 'what's left to walk' — can you say in a line why it's the same as a...
Fine-tuned model response: <start_of_turn>model
[4,2]. What did you see? What did you see was the identity matrix for a transformation? Picture it like a key that fits every lock. Now apply it to the identity matrix: what's the identity transformation for the vector [3,2]? What would you get?<end_of_turn>
--------------------------------------------------
Prompt: I'm stuck on what linear transformations is even for.
Sample of an expected response: Picture a rubber sheet pinned at the center that can stretch or rotate but never tear. Try columns [...
Fine-tuned model response: <start_of_turn>model
Think of a transformation that only ever makes one thing bigger or smaller, and nothing else. Try [1,0] and [0,2] — 1·0 + 0·2. What do you get?<end_of_turn

> **Note: Evaluating conversational models**
>
> Conversational models can be evaluated through a mix of automated metrics and human judgment, depending on how open-ended or task-focused the system is. **Human evaluation remains the gold standard for conversational models**, as it captures qualities like fluency, helpfulness, tone, and factual accuracy that automated metrics often miss. Human evaluators typically assess whether the model's replies are relevant, natural, and contextually appropriate.
>

## Future improvements

List of ways to improve the model's conversational abilities

* **Iterate systematically**:treat hyperparameters (rank, learning
  rate, epochs), the dataset composition, the base model, and even the
  data pipeline itself as things to experiment with. Each of these was set here based on reasoning rather than systematic comparison, and any could plausibly be improved with another pass.
* **Log experiments**: track each run's configuration and results
  (even a simple spreadsheet of hyperparameters + loss + sample outputs
  works) so different iterations can be compared directly.
* **Add automatic metrics alongside human review**: metrics like BLEU
  and ROUGE compare generated text against a reference answer via word/
  phrase overlap. They're imperfect for open-ended tutoring responses
  (many different phrasings can be equally good), but they're fast and
  consistent, making them useful for quickly tracking whether a change
  helped or hurt across many examples, without needing to manually read
  every output.
* **Add validation loss tracking during training**: Train loss alone can't reveal overfitting, since it only measures fit to data the model is already training on. Validation loss per epoch is needed to see whether unseen-data performance is actually improving or starting to decline, and to select the best epoch objectively rather than defaulting to the last one trained.
* **Re-test with a larger, stratified generalization sample**: this
  project's generalization check used a small qualitative sample; a
  larger sample stratified across concepts and trained behaviors (e.g.
  wrong-answer correction, multi-turn scaffolding) would give a more
  reliable picture of what actually generalizes.
